# Template

In [1]:
import polars as pl

import src.social_groups.polars_columns as plc
from social_groups.analysis.defs.notebooks.definitions import register_materialization
from social_groups.analysis.polars_transformations import make_group_constellation
from social_groups.analysis.polars_transformations.apply_parsing_and_group_decision import (
    apply_parsing_and_group_decision,
)
from social_groups.reporting.group_reply import (
    GroupReplyAggregator,
    MajorityVote,
)
from social_groups.reporting.parsing import (
    AnswerComparer,
    AnswerOptions,
    AnswerParser,
)

In [2]:
parser = AnswerParser(AnswerOptions.letters_A_to_J)
group_reply = GroupReplyAggregator(MajorityVote())
comparer = AnswerComparer(
    AnswerOptions.letters_A_to_J, triple_underscore_handling="wrong"
)

In [3]:
from social_groups.analysis.definitions import defs

changed_prompt: pl.DataFrame = defs().load_asset_value("changed_prompt_mad")
original_prompt: pl.DataFrame = defs().load_asset_value("hetero_mad")

/Users/philipp/Documents/Studium/Informatik/Masterthesis/Repository/.venv/lib/python3.11/site-packages/dagster/_config/pythonic_config/typing_utils.py:111: UserWarning: Field name "extension" in "PolarsParquetIOManager" shadows an attribute in parent "BasePolarsUPathIOManager"
  return super().__new__(cls, name, bases, namespaces, **kwargs)
2026-03-05 09:18:21 +0800 - dagster - DEBUG - system - Loading file from: /Users/philipp/Documents/Studium/Informatik/Masterthesis/Repository/results/analysis/dagster/changed_prompt_mad.parquet using PolarsParquetIOManager...
2026-03-05 09:18:21 +0800 - dagster - DEBUG - system - Loading file from: /Users/philipp/Documents/Studium/Informatik/Masterthesis/Repository/results/analysis/dagster/hetero_mad.parquet using PolarsParquetIOManager...


In [4]:
data = (
    apply_parsing_and_group_decision(
        original_prompt.with_columns(make_group_constellation()),
        parser,
        comparer,
        group_reply,
    )
    .group_by("group_constellation")
    .agg(pl.col("is_correct").mean().alias(plc.accuracy + "_original_mad"))
    .join(
        (
            apply_parsing_and_group_decision(
                changed_prompt, parser, comparer, group_reply
            )
            .group_by("group_constellation")
            .agg(pl.col("is_correct").mean().alias(plc.accuracy + "_changed_mad"))
        ),
        on="group_constellation",
    )
    .with_columns(
        pl.exclude("group_constellation", plc.accuracy + "_original_mad")
        .sub(pl.col(plc.accuracy + "_original_mad"))
        .name.prefix("delta_")
    )
    .sort("delta_accuracy_changed_mad", plc.group_constellation, descending=True)
)


register_materialization(
    "changed_prompts_mad_evaluation_table",
    data,
    "Accuracy of different group constellations using new prompts compared to the baseline MAD approach.",
)

data

group_constellation,accuracy_original_mad,accuracy_changed_mad,delta_accuracy_changed_mad
str,f64,f64,f64
"""HL""",0.53,0.47,-0.06
"""HM""",0.65,0.65,0.0
"""MMM""",0.58,0.61,0.03
"""HH""",0.77,0.69,-0.08
"""HHM""",0.7,0.7,0.0
…,…,…,…
"""LMM""",0.64,0.64,0.0
"""HLM""",0.67,0.68,0.01
"""LL""",0.28,0.34,0.06


Probing the Chats, there is still no coherence to actually "reason" about the answers given and providing feedback.
It is more like a

Probing the chats, there is still no "reasoning about former answers", they basically answer the same question again, with more stuff in the context.

Can we somehow show that this is the case?
 -> MAD = more context, answer again?